In [35]:
import os
import random
import warnings

import numpy as np
import pandas as pd
import torch

from datasets import load_dataset, Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Torch version: 2.5.1+cu121
CUDA available: True
GPU name: NVIDIA GeForce RTX 4070 Ti SUPER


In [36]:
def read_indices(path):
    with open(path, "r", encoding="utf-8") as f:
        return [int(line.strip()) for line in f if line.strip()]

In [ ]:
dataset = load_dataset("Tobi-Bueck/customer-support-tickets")
df = dataset["train"].to_pandas()

train_idx = read_indices("data/train_idx.txt")
val_idx = read_indices("data/val_idx.txt")
test_idx = read_indices("data/test_idx.txt")

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (49412, 16)
val: (6176, 16)
test: (6177, 16)


In [38]:
def normalize_text_col(s):
    s = s.fillna("").astype(str)
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

def make_text(df):
    subject = normalize_text_col(df["subject"])
    body = normalize_text_col(df["body"])
    return (subject + " [SEP] " + body).str.strip()

train_df["text"] = make_text(train_df)
val_df["text"] = make_text(val_df)
test_df["text"] = make_text(test_df)

In [46]:
train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["type"] = train_df["type"].fillna("Unknown")
val_df["type"] = val_df["type"].fillna("Unknown")
test_df["type"] = test_df["type"].fillna("Unknown")

In [47]:
from sklearn.preprocessing import LabelEncoder

queue_le = LabelEncoder()
priority_le = LabelEncoder()
type_le = LabelEncoder()

train_df["label_queue"] = queue_le.fit_transform(train_df["queue"])
val_df["label_queue"] = queue_le.transform(val_df["queue"])
test_df["label_queue"] = queue_le.transform(test_df["queue"])

train_df["label_priority"] = priority_le.fit_transform(train_df["priority"])
val_df["label_priority"] = priority_le.transform(val_df["priority"])
test_df["label_priority"] = priority_le.transform(test_df["priority"])

train_df["label_type"] = type_le.fit_transform(train_df["type"])
val_df["label_type"] = type_le.transform(val_df["type"])
test_df["label_type"] = type_le.transform(test_df["type"])

num_labels_queue = len(queue_le.classes_)
num_labels_priority = len(priority_le.classes_)
num_labels_type = len(type_le.classes_)

print(num_labels_queue, num_labels_priority, num_labels_type)

52 5 5


In [48]:
from datasets import Dataset

train_hf = Dataset.from_pandas(
    train_df[["text", "label_queue", "label_priority", "label_type"]],
    preserve_index=False
)
val_hf = Dataset.from_pandas(
    val_df[["text", "label_queue", "label_priority", "label_type"]],
    preserve_index=False
)
test_hf = Dataset.from_pandas(
    test_df[["text", "label_queue", "label_priority", "label_type"]],
    preserve_index=False
)

In [49]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-multilingual-cased"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_tokenized = train_hf.map(tokenize_function, batched=True)
val_tokenized = val_hf.map(tokenize_function, batched=True)
test_tokenized = test_hf.map(tokenize_function, batched=True)

Map:   0%|          | 0/49412 [00:00<?, ? examples/s]

Map:   0%|          | 0/6176 [00:00<?, ? examples/s]

Map:   0%|          | 0/6177 [00:00<?, ? examples/s]

In [50]:
import torch
import torch.nn as nn
from transformers import AutoModel, PreTrainedModel, AutoConfig

class MultiTaskTransformer(nn.Module):
    def __init__(
        self,
        model_name,
        num_labels_queue,
        num_labels_priority,
        num_labels_type,
        dropout=0.1,
    ):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(dropout)

        self.queue_classifier = nn.Linear(hidden_size, num_labels_queue)
        self.priority_classifier = nn.Linear(hidden_size, num_labels_priority)
        self.type_classifier = nn.Linear(hidden_size, num_labels_type)

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        label_queue=None,
        label_priority=None,
        label_type=None,
    ):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # Для DistilBERT берем hidden state [CLS]-позиции
        pooled_output = outputs.last_hidden_state[:, 0]
        pooled_output = self.dropout(pooled_output)

        logits_queue = self.queue_classifier(pooled_output)
        logits_priority = self.priority_classifier(pooled_output)
        logits_type = self.type_classifier(pooled_output)

        return {
            "logits_queue": logits_queue,
            "logits_priority": logits_priority,
            "logits_type": logits_type,
        }

In [51]:
from transformers import Trainer
import torch.nn.functional as F

class MultiTaskTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        label_queue = inputs.pop("label_queue")
        label_priority = inputs.pop("label_priority")
        label_type = inputs.pop("label_type")

        outputs = model(**inputs)

        loss_queue = F.cross_entropy(outputs["logits_queue"], label_queue)
        loss_priority = F.cross_entropy(outputs["logits_priority"], label_priority)
        loss_type = F.cross_entropy(outputs["logits_type"], label_type)

        loss = 0.70 * loss_queue + 0.15 * loss_priority + 0.15 * loss_type

        return (loss, outputs) if return_outputs else loss

In [52]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics_multitask(eval_pred):
    predictions, labels = eval_pred

    # predictions будет tuple/list из трех массивов logits
    logits_queue, logits_priority, logits_type = predictions

    labels_queue = labels[0]
    labels_priority = labels[1]
    labels_type = labels[2]

    preds_queue = np.argmax(logits_queue, axis=1)
    preds_priority = np.argmax(logits_priority, axis=1)
    preds_type = np.argmax(logits_type, axis=1)

    macro_f1_queue = f1_score(labels_queue, preds_queue, average="macro")
    acc_queue = accuracy_score(labels_queue, preds_queue)
    acc_priority = accuracy_score(labels_priority, preds_priority)
    acc_type = accuracy_score(labels_type, preds_type)

    final_score = (
        0.70 * macro_f1_queue
        + 0.15 * acc_priority
        + 0.15 * acc_type
    )

    return {
        "macro_f1_queue": macro_f1_queue,
        "accuracy_queue": acc_queue,
        "accuracy_priority": acc_priority,
        "accuracy_type": acc_type,
        "final_score": final_score,
    }

In [53]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [54]:
metric_for_best_model="final_score"

In [55]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../artifacts/multitask_transformer",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="final_score",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    label_names=["label_queue", "label_priority", "label_type"],
)

In [56]:
model = MultiTaskTransformer(
    model_name=MODEL_NAME,
    num_labels_queue=num_labels_queue,
    num_labels_priority=num_labels_priority,
    num_labels_type=num_labels_type,
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [57]:
trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_multitask,
)

In [58]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1 Queue,Accuracy Queue,Accuracy Priority,Accuracy Type,Final Score
1,1.545208,1.188060,0.733253,0.511496,0.504858,0.831930,0.713796
2,1.089899,1.061669,0.818889,0.566548,0.534488,0.844236,0.780031
3,0.937443,1.028881,0.843467,0.588407,0.551652,0.848931,0.800515


TrainOutput(global_step=9267, training_loss=1.190850050919931, metrics={'train_runtime': 463.3284, 'train_samples_per_second': 319.937, 'train_steps_per_second': 20.001, 'total_flos': 0.0, 'train_loss': 1.190850050919931, 'epoch': 3.0})

In [59]:
test_output = trainer.predict(test_tokenized)

In [60]:
logits_queue, logits_priority, logits_type = test_output.predictions
labels_queue, labels_priority, labels_type = test_output.label_ids

preds_queue = np.argmax(logits_queue, axis=1)
preds_priority = np.argmax(logits_priority, axis=1)
preds_type = np.argmax(logits_type, axis=1)

test_macro_f1_queue = f1_score(labels_queue, preds_queue, average="macro")
test_acc_queue = accuracy_score(labels_queue, preds_queue)
test_acc_priority = accuracy_score(labels_priority, preds_priority)
test_acc_type = accuracy_score(labels_type, preds_type)

final_score = (
    0.70 * test_macro_f1_queue
    + 0.15 * test_acc_priority
    + 0.15 * test_acc_type
)

print("Test Macro-F1 (queue):", test_macro_f1_queue)
print("Test Accuracy (queue):", test_acc_queue)
print("Test Accuracy (priority):", test_acc_priority)
print("Test Accuracy (type):", test_acc_type)
print("Final Score:", final_score)

Test Macro-F1 (queue): 0.8379309050726952
Test Accuracy (queue): 0.5834547514974907
Test Accuracy (priority): 0.5457341751659381
Test Accuracy (type): 0.8381091144568561
Final Score: 0.7941281269943057
